# 04 - Gold Layer
## 01 - Gold Outputs

### Goal
- derive business indicators from silver integrated table
- silver_integrated is at grid event granularity - aggregate to region/day in gold
- compute real price volatility from raw silver market prices
- build gold tables and views using SQL
- log all outputs

In [0]:
%run ../01_setup/00_config

## Register source tables as temp views

In [0]:
from pyspark.sql import functions as F

integrated_df    = spark.table(silver_integrated_table)
market_prices_df = spark.table(silver_market_prices_table)
weather_df       = spark.table(silver_weather_table)

integrated_df.createOrReplaceTempView("silver_integrated_view")
market_prices_df.createOrReplaceTempView("silver_market_prices_view")
weather_df.createOrReplaceTempView("silver_weather_view")

print(f"silver_integrated rows:    {integrated_df.count()}")
print(f"silver_market_prices rows: {market_prices_df.count()}")
print(f"silver_weather rows:       {weather_df.count()}")

## Gold table - market volatility summary

Computed from raw silver market prices - one row per `event_date`/`region`.
Volatility is the standard deviation across `DAY_AHEAD`, `INTRADAY`, and `BALANCING` prices within each region/day.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_market_volatility_table} AS
WITH price_stats AS (
    SELECT
        event_date,
        region,
        ROUND(AVG(price_eur_mwh), 2)    AS avg_price_eur_mwh,
        ROUND(MIN(price_eur_mwh), 2)    AS min_price_eur_mwh,
        ROUND(MAX(price_eur_mwh), 2)    AS max_price_eur_mwh,
        ROUND(STDDEV(price_eur_mwh), 4) AS price_volatility,
        ROUND(SUM(volume_mwh), 2)       AS total_volume_mwh
    FROM silver_market_prices_view
    GROUP BY event_date, region
)
SELECT
    *,
    ROUND(max_price_eur_mwh - min_price_eur_mwh, 2) AS price_range,
    CASE
        WHEN price_volatility > 10 THEN 'HIGH'
        WHEN price_volatility > 5  THEN 'MEDIUM'
        ELSE 'LOW'
    END AS volatility_band
FROM price_stats
ORDER BY event_date, region
""")

print(f"Gold table saved: {gold_market_volatility_table}")
display(spark.table(gold_market_volatility_table).limit(10))

## Gold table - regional operations summary

Aggregates grid events to one row per `event_date`/`region`.
Joins market volatility to bring in real price metrics.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_regional_operations_table} AS
SELECT
    i.event_date,
    i.region,
    FIRST(i.country_code)                     AS country_code,
    FIRST(i.operating_zone)                   AS operating_zone,

    -- market indicators from volatility table
    v.avg_price_eur_mwh,
    v.total_volume_mwh,
    v.price_volatility,
    v.price_range,
    v.volatility_band,
    FIRST(i.has_high_price)                   AS has_high_price,

    -- weather indicators - already daily aggregates
    FIRST(i.avg_temperature_c)                AS avg_temperature_c,
    FIRST(i.avg_wind_speed_kmh)               AS avg_wind_speed_kmh,
    FIRST(i.total_precipitation_mm)           AS total_precipitation_mm,
    FIRST(i.has_severe_weather)               AS has_severe_weather,
    FIRST(i.wind_class)                       AS wind_class,

    -- event severity
    SUM(i.is_critical_event)                  AS critical_event_count,
    COUNT(i.event_id)                         AS total_events,
    ROUND(AVG(i.duration_minutes), 2)         AS avg_event_duration_minutes,

    -- derived scores
    CASE
        WHEN FIRST(i.has_severe_weather) = 1 THEN 2
        WHEN FIRST(i.avg_wind_speed_kmh) >= 60 THEN 1
        ELSE 0
    END AS weather_impact_score,

    CASE
        WHEN SUM(i.is_critical_event) > 0 AND FIRST(i.has_severe_weather) = 1 THEN 'HIGH'
        WHEN SUM(i.is_critical_event) > 0 OR FIRST(i.has_severe_weather) = 1  THEN 'MEDIUM'
        ELSE 'LOW'
    END AS operational_risk

FROM silver_integrated_view i
LEFT JOIN {gold_market_volatility_table} v
    ON i.event_date = v.event_date AND i.region = v.region
GROUP BY
    i.event_date, i.region,
    v.avg_price_eur_mwh, v.total_volume_mwh,
    v.price_volatility, v.price_range, v.volatility_band
ORDER BY i.event_date, i.region
""")

print(f"Gold table saved: {gold_regional_operations_table}")
display(spark.table(gold_regional_operations_table).limit(10))
dbutils.jobs.taskValues.set("gold_row_count", str(spark.table(gold_regional_operations_table).count()))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## Gold table - market summary

Market prices enriched with daily weather context. One row per `event_date`/`region`.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_market_summary_table} AS
WITH market_daily AS (
    SELECT
        event_date, region,
        ROUND(AVG(price_eur_mwh), 2)    AS avg_price_eur_mwh,
        ROUND(MIN(price_eur_mwh), 2)    AS min_price_eur_mwh,
        ROUND(MAX(price_eur_mwh), 2)    AS max_price_eur_mwh,
        ROUND(STDDEV(price_eur_mwh), 4) AS price_volatility,
        ROUND(SUM(volume_mwh), 2)       AS total_volume_mwh,
        MAX(is_high_price)              AS has_high_price
    FROM silver_market_prices_view
    GROUP BY event_date, region
),
weather_daily AS (
    SELECT
        event_date, region,
        ROUND(AVG(temperature_c), 2)    AS avg_temperature_c,
        ROUND(AVG(wind_speed_kmh), 2)   AS avg_wind_speed_kmh,
        ROUND(SUM(precipitation_mm), 2) AS total_precipitation_mm,
        MAX(is_severe_weather)          AS has_severe_weather,
        FIRST(wind_class)               AS wind_class
    FROM silver_weather_view
    GROUP BY event_date, region
)
SELECT
    m.event_date, m.region,
    m.avg_price_eur_mwh, m.min_price_eur_mwh, m.max_price_eur_mwh,
    ROUND(m.max_price_eur_mwh - m.min_price_eur_mwh, 2) AS price_range,
    m.price_volatility,
    CASE
        WHEN m.price_volatility > 10 THEN 'HIGH'
        WHEN m.price_volatility > 5  THEN 'MEDIUM'
        ELSE 'LOW'
    END AS volatility_band,
    m.total_volume_mwh, m.has_high_price,
    w.avg_temperature_c, w.avg_wind_speed_kmh,
    w.total_precipitation_mm, w.has_severe_weather, w.wind_class,
    CASE
        WHEN w.has_severe_weather = 1 THEN 2
        WHEN w.avg_wind_speed_kmh >= 60 THEN 1
        ELSE 0
    END AS weather_impact_score
FROM market_daily m
LEFT JOIN weather_daily w ON m.event_date = w.event_date AND m.region = w.region
ORDER BY m.event_date, m.region
""")
print(f"Saved: {gold_market_summary_table} - {spark.table(gold_market_summary_table).count()} rows")
display(spark.table(gold_market_summary_table).limit(5))

##Gold table - grid summary
Grid events aggregated to one row per event_date/region with operational risk.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_grid_summary_table} AS
SELECT
    event_date, region,
    FIRST(country_code)                   AS country_code,
    FIRST(operating_zone)                 AS operating_zone,
    COUNT(event_id)                       AS total_events,
    SUM(is_critical_event)                AS critical_event_count,
    ROUND(AVG(duration_minutes), 2)       AS avg_event_duration_minutes,
    FIRST(has_severe_weather)             AS has_severe_weather,
    CASE
        WHEN FIRST(has_severe_weather) = 1 THEN 2
        WHEN SUM(is_critical_event) >= 3   THEN 1
        ELSE 0
    END AS weather_impact_score,
    CASE
        WHEN SUM(is_critical_event) > 0 AND FIRST(has_severe_weather) = 1 THEN 'HIGH'
        WHEN SUM(is_critical_event) > 0 OR  FIRST(has_severe_weather) = 1 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS operational_risk
FROM silver_integrated_view
GROUP BY event_date, region
ORDER BY event_date, region
""")
print(f"Saved: {gold_grid_summary_table} - {spark.table(gold_grid_summary_table).count()} rows")
display(spark.table(gold_grid_summary_table).limit(5))

Databricks visualization. Run in Databricks to view.

## Gold table - weather impact summary
Weather observations aggregated to one row per event_date/region.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_weather_impact_table} AS
SELECT
    event_date, region,
    ROUND(AVG(temperature_c), 2)    AS avg_temperature_c,
    ROUND(MIN(temperature_c), 2)    AS min_temperature_c,
    ROUND(MAX(temperature_c), 2)    AS max_temperature_c,
    ROUND(AVG(wind_speed_kmh), 2)   AS avg_wind_speed_kmh,
    ROUND(MAX(wind_speed_kmh), 2)   AS max_wind_speed_kmh,
    ROUND(SUM(precipitation_mm), 2) AS total_precipitation_mm,
    MAX(is_severe_weather)          AS has_severe_weather,
    FIRST(wind_class)               AS dominant_wind_class,
    CASE
        WHEN MAX(is_severe_weather) = 1          THEN 2
        WHEN ROUND(AVG(wind_speed_kmh), 2) >= 60 THEN 1
        ELSE 0
    END AS weather_impact_score
FROM silver_weather_view
GROUP BY event_date, region
ORDER BY event_date, region
""")
print(f"Saved: {gold_weather_impact_table} - {spark.table(gold_weather_impact_table).count()} rows")
display(spark.table(gold_weather_impact_table).limit(5))

Databricks visualization. Run in Databricks to view.

## Gold table - operational KPI

One row per `event_date`/`region` showing % HIGH risk days and event counts.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_operational_kpi_table} AS
WITH daily_risk AS (
    SELECT
        event_date, region,
        operational_risk,
        critical_event_count,
        total_events,
        avg_event_duration_minutes,
        CASE WHEN operational_risk = 'HIGH'   THEN 1 ELSE 0 END AS is_high_risk,
        CASE WHEN operational_risk = 'MEDIUM' THEN 1 ELSE 0 END AS is_medium_risk,
        CASE WHEN operational_risk = 'LOW'    THEN 1 ELSE 0 END AS is_low_risk
    FROM {gold_grid_summary_table}
),
with_windows AS (
    SELECT
        *,
        SUM(is_high_risk) OVER (
            PARTITION BY region ORDER BY event_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_high_risk_days,
        COUNT(*) OVER (
            PARTITION BY region ORDER BY event_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_total_days
    FROM daily_risk
)
SELECT
    event_date,
    region,
    operational_risk,
    critical_event_count,
    total_events,
    avg_event_duration_minutes,
    is_high_risk,
    is_medium_risk,
    is_low_risk,
    ROUND(cumulative_high_risk_days / cumulative_total_days * 100, 2) AS pct_high_risk_days_cumulative
FROM with_windows
ORDER BY event_date, region
""")
print(f"Saved: {gold_operational_kpi_table} - {spark.table(gold_operational_kpi_table).count()} rows")
display(spark.table(gold_operational_kpi_table).limit(5))

Databricks visualization. Run in Databricks to view.

## Gold view - dashboard ready summary

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {gold_dashboard_view} AS
SELECT
    event_date,
    region,
    country_code,
    operating_zone,
    avg_price_eur_mwh,
    price_volatility,
    price_range,
    volatility_band,
    total_volume_mwh,
    avg_temperature_c,
    avg_wind_speed_kmh,
    has_severe_weather,
    weather_impact_score,
    critical_event_count,
    total_events,
    operational_risk
FROM {gold_regional_operations_table}
""")

print(f"Gold view created: {gold_dashboard_view}")
display(spark.table(gold_dashboard_view).limit(10))

Databricks visualization. Run in Databricks to view.

## Logging summary

In [0]:
print("=" * 60)
print("GOLD LAYER - RUN SUMMARY")
print("=" * 60)
print(f"silver_integrated:          {integrated_df.count()} rows")
print(f"silver_market_prices:       {market_prices_df.count()} rows")
print(f"silver_weather:             {weather_df.count()} rows")
print()
print(f"gold_market_volatility:     {spark.table(gold_market_volatility_table).count()} rows")
print(f"gold_regional_operations:   {spark.table(gold_regional_operations_table).count()} rows")
print(f"gold_market_summary:        {spark.table(gold_market_summary_table).count()} rows")
print(f"gold_grid_summary:          {spark.table(gold_grid_summary_table).count()} rows")
print(f"gold_weather_impact:        {spark.table(gold_weather_impact_table).count()} rows")
print(f"gold_operational_kpi:       {spark.table(gold_operational_kpi_table).count()} rows")
print()
print(f"Views: {gold_dashboard_view}")
print(f"Run date: {spark.sql('SELECT current_date()').collect()[0][0]}")
print("=" * 60)